In [ ]:
%matplotlib widget

In [ ]:
import hyperspy.api as hs
import matplotlib.pyplot as plt
import numpy as np

from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import root, least_squares

from scipy.ndimage import shift
from skimage.transform import AffineTransform, warp

from pathlib import Path

import importlib
import get_transforms
import get_masks
importlib.reload(get_transforms)
importlib.reload(get_masks)

In [ ]:
# Alle zspy-filene fra 002 - 023 = 22 filer
data_folder = Path("data")
files = sorted(data_folder.glob("*.zspy"))
original_images = {}
s_dir = {}

for file in files:
    s = hs.load(file, lazy=True)
    s_bs = s.sum(axis=(-1, -2))
    s_bs.compute()
    original_images[file.stem] = s_bs
    s_dir[file.stem] = s

#Sjekker om filene er prosessert: 
print(f"Processed {len(original_images)} files")

In [ ]:
registered_images, transformations, shifts = get_transforms.main(original_images)

In [ ]:
# Mask is not really dependent on transformation, 002 is just normalized
# Center is (row, column)
masks, center, radii = get_masks.main(original_images, registered_images)

In [ ]:
def find_root(data, mask):
    rows, cols = np.arange(data.shape[0]), np.arange(data.shape[1])
    field = np.stack((data.real, data.imag), axis=-1)
    f = RegularGridInterpolator((rows, cols), field)
    
    valid = mask[:-1, :-1] & mask[1:, :-1] & mask[:-1, 1:] & mask[1:, 1:]
    
    corners = np.stack((
        data[:-1, :-1], data[1:, :-1],
        data[:-1, 1:], data[1:, 1:]
    ))
    
    candidates = valid & (
        (corners.real.min(0) <= 0) & (corners.real.max(0) >= 0) &
        (corners.imag.min(0) <= 0) & (corners.imag.max(0) >= 0)
    )
    
    solutions = []
    for r, c in np.argwhere(candidates):
        sol = least_squares(
            lambda p: f(p).ravel(),
            (r + 0.5, c + 0.5),
            bounds=([r, c], [r + 1, c + 1])
        )
        solutions.append(sol)
    
    if not solutions:
        return (None, None)
        # raise RuntimeError("No zero-crossing candidate found inside the mask")
    
    result = min(solutions, key=lambda s: np.linalg.norm(s.fun))
    
    # print("row, col:", result.x)
    # print("x, y:", result.x[::-1])
    # print("residual:", np.linalg.norm(result.fun))
    return result.x

In [ ]:
def warp_data(data, transformation, shift_vec):
    data_warp = warp(
        data,
        inverse_map=AffineTransform(matrix = transformation.params),
        preserve_range=True
    )
    
    data_shift_x = shift(
        data_warp[:, :, 0],
        shift_vec
    )
    
    data_shift_y = shift(
        data_warp[:, :, 1],
        shift_vec
    )
    
    return (
        data_shift_x
        + 1j * data_shift_y
    )

In [ ]:
data_folder = Path("data_shift")
files = sorted(data_folder.glob("*.hspy"))

s_bs_rot_dir = {}
for file in files:
    s_bs_rot_dir[file.stem] = hs.load(file)

In [ ]:
result = [[(None, None) for _ in range(len(s_dir))] for _ in range(len(masks))]

for s_name in s_dir:
    print(s_name)
    s = s_dir[s_name]
    data = s_bs_rot_dir[s_name].data
    data = warp_data(data, transformations[s_name], shifts[s_name])
    for i, mask in enumerate(masks):
        # Janky hack mate
        result[i][int(s_name)-2] = find_root(data, mask)

In [ ]:
result[16][2]